In [22]:
# Necessary libraries and useful parameters
import numpy as np
import os

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.colorbar as cb

import scipy.sparse as sp
import warnings
warnings.simplefilter("ignore", RuntimeWarning)

newparams = {'font.family': 'cmr10', 'mathtext.fontset': 'cm',
             'axes.grid': False, 'axes.labelsize': 22,
             'xtick.labelsize': 18, 'ytick.labelsize': 18,
             'legend.fontsize': 18, 'axes.titlesize': 20,
             'figure.figsize': (9,6), 'lines.linewidth': 2.5,
             'axes.formatter.use_mathtext': True}
plt.rcParams.update(newparams)
%matplotlib inline

In [23]:
# Color maps
concentration_colors = ["#00274D", "#1B4F72", "#CFA6A6", "#CD5C5C", "#8B3A3A", "#641010"]
concentration_cmap = mcolors.LinearSegmentedColormap.from_list("custom_blue_map", concentration_colors)

deposition_colors = ["#F6CCCC", "#CD5C5C", "#641010"]
deposition_cmap = mcolors.LinearSegmentedColormap.from_list("custom_reds", deposition_colors)

In [24]:
# Global model parameters
domain = ((0, 1), (0, 1), (0, 1))
M, K = 200, 100
h, ht = 1/M, 1/K

z = np.linspace(domain[0][0], domain[0][1], M+1)
r = np.linspace(domain[1][0], domain[1][1], M+1)
t = np.linspace(domain[2][0], domain[2][1], K+1)

grid = (z, r)
ZZ, RR = np.meshgrid(z, r) 

In [25]:
# Scaled block matrix coefficients
def matrix_coeffs(params):
    R, L, E, U, D, kA = params

    δ = ht/h**2 * D
    γ = δ * (R/L)**2
    ξ = ht/h * D*(E-1)/r
    𝜗 = ht/h * U*(1-r**2)
    ω = ξ + 𝜗 + 2*(γ + δ) + 1

    return δ, γ, ξ, 𝜗, ω

In [26]:
def coeff_matrix(M:int, coeffs) -> sp.csr_matrix:
    '''
    Create the block tridiagonal matrix A to solve AC^k+1 + g = -C^k.
    Input:
        M: number of grid points
    '''
    Z = M*(M-1)
    δ, γ, ξ, 𝜗, ω = coeffs

    ll = np.repeat(δ + ξ[1:-1], M)      # Lowest diag
    l = np.repeat(γ + 𝜗[1:-1], M) # Next lowest diag
    d = np.repeat(ω[1:-1], M)            # Main diag 
    u = np.full(Z, γ)                    # Next highest diag
    uu = np.full(Z, δ)                   # Highest diag
    
    l[M-1::M], u[M-1::M] = 0, 0 # Every M'th element is zero in l and d
    
    A = sp.diags([ll, l, -d, u, uu], [-M, -1, 0, 1, M], (Z, Z), format='csr')

    return A

In [27]:
def boundary_vector(grid:tuple, k:int, qk:np.ndarray, Ck:np.ndarray, BC:tuple, M:int, coeffs, params) -> np.ndarray:
  ''' 
  Create the boundary vector g to solve AC^k+1 + g = -C^k.
  Input:
    grid: (r, z) grids in r- and z directions
       k: the previous iteration step
      Ck: the previous concentration matrix
      BC: (g1, g2, g3) boundary condition functions
       M: number of grid points
  '''
  z, r = grid
  g1, g2, g3 = BC
  δ, γ, ξ, 𝜗, ω = coeffs
  
  # Construct a matrix containing boundary contributions.
  G = np.zeros((M-1, M))
  G[::-1, 0] += (γ+𝜗[1:-1]) * g1(r[1:-1], t[k+1]) # Left contribution
  G[-1, :] += (δ+ξ[1]) * g2(z[1:], t[k+1])              # Bottom contribution
  G[0, :] += δ * g3(qk[1:], Ck[-2, 1:], params)                   # Top contribution

  return G[::-1].ravel() # return G as a vector

In [28]:
def concentration_scheme(grid:tuple, A:np.ndarray, g:np.ndarray, k:int, qk:np.ndarray, Ck:np.ndarray, BC:tuple, M:int, params) -> np.ndarray:
    '''
    Calculate the next concentration matrix C^k+1.
    Input:
        grid: (r, z) grids in r- and z directions
           A: the coefficient matrix
           k: the previous iteration step
          Ck: the previous concentration matrix
           g: the boundary vector
          BC: (g1, g2, g3) boundary condition functions
           M: number of grid points
    '''
    z, r = grid
    g1, g2, g3 = BC
   
    C = np.zeros((M+1, M+1))
    C[::-1, 0] = g1(r, t[k+1]) # Left boundary
    C[-1, :] = g2(z, t[k+1])   # Bottom boundary  
    C[0, 1:] = g3(qk[1:], Ck[-2, 1:], params) # Top boundary

    C_interior = sp.linalg.spsolve(A, -(g + Ck[1:-1, 1:][::-1].ravel()))
    C[1:-1, 1:] = np.reshape(C_interior, (M-1, M))
   
    return C

In [29]:
def RK4(f, q, C, params):
    ''' 
    Solve a set of ODE's using the RK4-method.
    Input:
        f: the right hand side of the differential equations. Here: The Langmuir model
        q: initial values
        C: concentration matrix
        params: parameters needed by f. Here: [E, U, D, kA]
    '''
    k1 = f(q, C, params)
    k2 = f(q+ht*k1/2, C, params)
    k3 = f(q+ht*k2/2, C, params)
    k4 = f(q+ht*k3, C, params)
    
    return q + h/6*(k1 + 2*(k2 + k3) + k4)

In [30]:
# Initial condition
def f(z, r):                   
    return np.zeros_like(ZZ)   
                                                         
# Boundary conditions
def i(r, t): # inlet
    return 1-r**2  

def e(z, t): # electric
    return 0

def langmuir(q, C, params): # RHS of langmuir model to be solved with RK4
    R, L, E, U, D, kA = params
    return kA*C*(1-q)

def d(qk, C, params): # deposition (discretized)
    R, L, E, U, D, kA = params
    return C / (1 - h*((E*R)/r[-1] + kA/D *(1-qk)))

In [31]:
# Simulator functions

def contour_plot(Z, R, C, dCz=None, dCr=None, skip=12, output_file=None):
    # Plot base concentration contour
    contour = plt.contourf(Z, R, C, levels=200, cmap=concentration_cmap)
    cbar = plt.colorbar(contour, label=r'$c/c_{\mathrm{max}}$')

    if dCz is not None and dCr is not None:
        Z_skip   = Z[::skip, ::skip]   # Subsample for clarity
        R_skip   = R[::skip, ::skip]
        dCz_skip = dCz[::skip, ::skip]
        dCr_skip = dCr[::skip, ::skip]

        mag = np.sqrt(dCz_skip**2 + dCr_skip**2)

        # Enhance contrast in lengths using exponent
        exponent = 0.04
        mag_scaled = mag**exponent
        mag_scaled /= np.max(mag_scaled)  # Normalize to [0, 1]

        dCz_scaled = dCz_skip * mag_scaled / mag  # Scale vector components by adjusted magnitude
        dCr_scaled = dCr_skip * mag_scaled / mag

        arrow_length = 0.1      # Rescale to desired max arrow length
        dCz_final = dCz_scaled * arrow_length
        dCr_final = dCr_scaled * arrow_length

        plt.quiver(Z_skip, R_skip, dCz_final, dCr_final, width=0.003, color='slategrey', scale=1.5, scale_units='xy')

    plt.xlabel('$z/L$')
    plt.ylabel('$r/R$')
    plt.gca().set_aspect(aspect=0.6, adjustable='box')
    if output_file is not None:
        plt.savefig(output_file, bbox_inches='tight')
    plt.show()

In [ ]:
def simulate_concentration(R, L, E, U, D, kA, label=None, show_contour=False, snapshot_times=(0, 0.04, 0.14, 0.49), prefix=None):
    params = [R, L, E, U, D, kA]
    coeffs = matrix_coeffs(params)
    A = coeff_matrix(M, coeffs)
    BC = (i, e, d)

    Ck = f(ZZ, RR)         # Initial concentration matrix
    qk = np.zeros(M+1)     # Initial wall surface coverage
    Cwall = np.zeros((K, M+1))

    # Convert fractions of time domain into indices
    snapshot_indices = set(np.round(np.array(snapshot_times) * (K - 1)).astype(int))
    numeric = 0
    
    for k, time in enumerate(t[:-1]):
        g = boundary_vector(grid, k, qk, Ck, BC, M, coeffs, params)
        C = concentration_scheme(grid, A, g, k, qk, Ck, BC, M, params)
        dCr, dCz = np.gradient(C, h, h)

        Cwall[k] = C[0]
        q = RK4(langmuir, qk, C[-1, :], params)

        if show_contour and k in snapshot_indices:
            plt.figure()
            plt.title(f'$t/T = {time+ht:.2f}$')
            if prefix is not None:
                os.makedirs(os.path.dirname(f"output/{prefix}/"), exist_ok=True)
            output_file = None if prefix is None else f"output/{prefix}/{numeric:04d}.png"
            contour_plot(ZZ, RR, C, -dCz, -dCr, output_file=output_file) # Note: negative gradient is flow direction
            numeric += 1

        Ck = C
        qk = q

    return Cwall

In [33]:
def simulate_deposition(Cwall, profiles=20, output_file=None):
    plt.figure()

    # Select "profiles" evenly spaced indices from the time steps
    profile_indices = np.round(np.linspace(0, K - 1, profiles)).astype(int)

    for j, k in enumerate(profile_indices):
        color = deposition_cmap(j / max(1, profiles - 1))  # Normalize color mapping
        plt.plot(z, Cwall[k], color=color)

    plt.xlabel('$z/L$')
    plt.ylabel(r'$c/c_{\mathrm{max}}$')
    plt.grid(True)

    sm = plt.cm.ScalarMappable(cmap=deposition_cmap, norm=plt.Normalize(vmin=t[0], vmax=t[-1]))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=plt.gca())
    cbar.set_label('$t/T$')
    if output_file is not None:
        plt.savefig(output_file, bbox_inches='tight')
    plt.show()

In [ ]:
# Example 1: Varying E, contour plots, deposition profile
R = 0.1
L = 10*R
U, D, kA = 4, 1, 1
E_list = [1, 2, 3]
for E in E_list:
    Cwall = simulate_concentration(R, L, E, U, D, kA, show_contour=True, prefix=f"E_{E}")
    simulate_deposition(Cwall)

In [ ]:
# Example 2: Varying U, contour plots, deposition profile
R = 0.1
L = 10*R
E, D, kA = 3, 1, 1
U_list = [2, 4, 8]
for U in U_list:
    Cwall = simulate_concentration(R, L, E, U, D, kA, show_contour=True,  prefix=f"U_{U}")
    simulate_deposition(Cwall)

In [ ]:
# Example 3: Varying D, contourplots, deposition profile
R = 0.1
L = 10*R
E, U, kA = 3, 4, 1
D_list = [0.5, 1, 2.5]
for D in D_list:
    Cwall = simulate_concentration(R, L, E, U, D, kA, show_contour=True, prefix=f"D_{D}")
    simulate_deposition(Cwall)

In [ ]:
# Example 4: Varying kA, contour plots, deposition profile
R = 0.1
L = 10*R
E, U, D = 3, 4, 1
kA_list = [0.5, 2, 5]
for kA in kA_list:
    Cwall = simulate_concentration(R, L, E, U, D, kA, show_contour=True, prefix=f"kA_{kA}")
    print(Cwall)
    simulate_deposition(Cwall)

In [ ]:
# Example 5.1: Holding tube radius constant, varying length
R = 0.1
E, U, D, kA = 3, 4, 1, 1
L_list = [2*R, 5*R, 15*R, 25*R]
for L in L_list:
    Cwall = simulate_concentration(R, L, E, U, D, kA)
    simulate_deposition(Cwall)

In [ ]:
from gif_gen import gen_gif

base_name = "./output"
variable = "kA"
for val in kA_list:
    gen_gif(folder=f"{base_name}/{variable}_{val}", output_file=f"{variable}_{val}_fds.gif")

In [ ]:
def simulate_deposition_aggregate(Cwalls):
    fig, ax = plt.subplots()
    ax.set_xlabel('$z/L$')
    ax.set_ylabel(r'$c/c_{\mathrm{max}}$')
    ax.grid(True)
    # Could turn the colorbar into the quality of the solution.
    # sm = plt.cm.ScalarMappable(cmap=deposition_cmap, norm=plt.Normalize(vmin=t[0], vmax=t[-1]))
    # sm.set_array([])
    # cbar = fig.colorbar(sm, ax=plt.gca())
    # cbar.set_label('$t/T$')
    for i in range(len(Cwalls)):
        if i == len(Cwalls) - 1:
            ax.plot(z, Cwalls[i], color='red', linewidth=3, marker='o', label='Final profile')
        else:
            ax.plot(z, Cwalls[i], color='black')
    ax.legend()
    ax.set_title("Single Objective Deposition Optimization")
    fig.savefig("single_obj.png")
       

## Bayesian Optimization

In [ ]:
# # Optimize E, U, D to get the most uniform spread of deposition.
# # Need to create an objective function related to spread.
# # Later, bring in the efficiency of filtration as well as the energy consumption
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.plots import plot_convergence
# Example 1: Varying E, contour plots, deposition profile

def objective_func(deposition_c):
    """
        deposition_c - The concentration along the deposition wall.
    """
    return 10*np.var(deposition_c) + 1 / sum(deposition_c)

global Cwalls
Cwalls = []

def evaluate_sim(search_space, show_contour=False, prefix=None):
    global Cwalls
    E, U, D, kA = search_space
    Cwall = simulate_concentration(R, L, E, U, D, kA, show_contour=show_contour, prefix=prefix)
    Cwalls.append(Cwall)
    simulate_deposition(Cwall)
    # simulate_deposition_aggregate(Cwall, fig, ax)
    final_profile = Cwall[-1, :]
    return objective_func(final_profile)


In [ ]:
global Cwalls
Cwalls = []

R = 0.1
L = 10*R

search_space = [
        Real(1.0, 3.0, name='E'),  
        Real(1.0, 10.0, name='U'),  
        Real(1.0, 2.0, name='D'),  
        Real(1.0, 5.0, name='kA')
]

result = gp_minimize(evaluate_sim,      
                     search_space,                   
                     n_calls=20,              
                     random_state=42) 

print(result)

In [ ]:
# print(result['x_iters'])
# print(Cwalls)
best_result = result['x']
print(best_result)
index_to_remove = -1
for i in range(len(result['x_iters'])):
    if result['x_iters'][i] == best_result:
        index_to_remove = i
        break

best_val = Cwalls.pop(index_to_remove)
Cwalls.append(best_val)
final_profiles = [cwall[-1] for cwall in Cwalls]

In [ ]:
simulate_deposition_aggregate(final_profiles)

In [ ]:
evaluate_sim([3.0, 9.357549135260617, 1.0, 1.3683146573615916], show_contour=True, prefix="best")

In [ ]:
from gif_gen import gen_gif

base_name = "./output"

gen_gif(folder=f"{base_name}/best", output_file=f"best_fds.gif")

In [ ]:
plot_convergence(result)